# API calls

What is an API call? - look it up what could it be useful for in your work?

It is important to point out that public APIs like PubChem and ChEMBL can collect data on the queries run, so should always be used with caution with industry data 

Example code below is CAS and pKa lookups - either modify this code or create your own code using the skills learnt in this course

In [ ]:

import re

import pubchempy as pcp

def cas_check(potential: str) -> bool:
    """
    Checks whether a string is a valid CAS number by verifying its check digit.

    Args:
        potential (str): The CAS number string to check.

    Returns:
        bool: True if the string has a valid CAS check digit, False otherwise.
    """
    short_cas = potential[:-1].replace('-', '') # CAS without hyphens and check digit
    cas_length = len(short_cas) # Length without hyphens and check digit

    # Calculate the sum for the check digit
    check_sum = 0
    for n, d in enumerate(short_cas):
        try:
            check_sum += float(d) * (cas_length - n)
        except ValueError: # Cannot convert to float
            return False
        
    check_digit = int(check_sum % 10) # What check digit should equal
    if str(check_digit) == potential[-1]:
        return True
    else:
        return False

# Function to determine whether a string is a valid CAS number
def is_cas(potential: str) -> bool:
    """
    Determines whether a string is a valid CAS number (format and check digit).

    Args:
        potential (str): The string to check for CAS number validity.

    Returns:
        bool: True if the string is a valid CAS number, False otherwise.
    """
    if not isinstance(potential, str):
        return False
    
    # First, check whether potential has CAS-like format
    x = re.match('\d{2,7}-\d\d-\d', potential)
    if x is None: # Not a CAS-like string
        return False

    # Now check whether the CAS digit is valid
    if cas_check(potential):
        return True # Valid CAS number
    else:
        return False # Check digit is wrong

def name_to_cas(name: str) -> str:
    """
    Looks up the CAS number for a given chemical name using PubChemPy.

    Args:
        name (str): The chemical name to look up.

    Returns:
        str: The CAS number if found, otherwise 'not found'.
    """
    compounds = pcp.get_compounds(name, 'name')
    if len(compounds) == 0:
        return 'not found'

    cpd = compounds[0]
    if cpd is not None and hasattr(cpd, 'cid'):
        cas_found = False  # Default value
        for synonym in cpd.synonyms:
            cas_found = is_cas(synonym)
            if cas_found:
                return synonym

    return 'not found'

# A test
name = 'acetic acid'
result = name_to_cas(name)
print(f'CAS number for {name}: {result}')


In [ ]:
'''
Acknowledgements: Help for the initial version of this many moons ago, from Dr Khoi Van (currently at Amazon)
'''
__author__ = "Nessa Carson"
__copyright__ = "Copyright 2019, 2026"
__email__ = "nessa.carson@astrazeneca.com"

import json
from pathlib import Path
import urllib
import xml.etree.ElementTree as ET

import pubchempy as pcp
import requests

def get_proxies(proxy_file: Path) -> dict[str, str]:
    """
    Loads proxy settings from a JSON file and installs them for urllib requests.

    Args:
        proxy_file (Path): Path to the JSON file containing proxy settings.

    Returns:
        dict[str, str]: Dictionary of proxies if the file exists and is valid, otherwise an empty dictionary.
    """
    if proxy_file.is_file():
        with open(proxy_file, 'rt') as fin:
            proxies = json.load(fin)
        proxy_support = urllib.request.ProxyHandler(proxies)
        opener = urllib.request.build_opener(proxy_support)
        urllib.request.install_opener(opener)
        return proxies

    else:
        return {}

def pka_lookup(cas: str, print_output: bool=True) -> str | None:
    '''
    Returns the pKa for a substance given its CAS number or chemical name.

    Args:
        cas (str): The CAS number (or chemical name) of the substance.
        print_output (bool): Flag indicating whether to print the pKa result. Defaults to True.

    Returns:
        str | None: The pKa value as a string, or None if not found in the database.

    Note: 
        A string is returned as there is often extra information included, eg: temperature. This would 
        require further parsing to return a float.
    '''
    cids = pcp.get_cids(cas, 'name')
    if not cids: # Check whether cids is empty or None
        if print_output:
            print(f'Compound {cas} not found')
        return None
    
    # Take only the first CID from the list (assumes it's the most relevant match)
    cid = cids[0]
    pka_lookup_xml = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/XML?heading=Dissociation+Constants'

    try:
        r = requests.get(pka_lookup_xml, headers=headers, timeout=10, proxies=proxies)
    except requests.exceptions.RequestException as error:
        if print_output:
            print(f'Error retrieving pKa: {str(error)}')
        return None
        
    pka_result = None # Default value if pKa not found
    tree = ET.fromstring(r.text)
    for node in tree.iter('{http://pubchem.ncbi.nlm.nih.gov/pug_view}String'):
        pka_result = node.text
    if print_output:
        print(pka_result if pka_result else 'pKa not found in database.')

    return pka_result

headers = {
    'user-agent': 'Mozilla/5.0 (X11; CentOS; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/73.0.3683.75 Safari/537.36',
    'Connection': 'close'}
proxy_file = Path('proxies.dat')
proxies = {} # Default global value

if __name__ == '__main__':
    # Load proxies
    proxies = get_proxies(proxy_file)
    # Request user input for one CAS number/chemical name
    cas_number = input("Enter CAS number or chemical name: ")
    # Return the pKa from PubChem
    pka_lookup(cas_number)

''' Users to think about: how would you modify this script to take a list of CAS numbers from a file, or allow constant user input in series?'''